# Fase 6.2 — Dashboard Interactivo

**Pregunta de negocio:** Dashboard integral para monitoreo de operaciones de taxi

**Objetivo:** Construir un dashboard interactivo con Plotly que permita explorar los datos
de NYC Taxi Trips 2015 de manera visual, incluyendo KPIs, mapas de calor, series temporales,
deteccion de anomalias y mapa geografico.

**Dataset:** `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`

---

## 0. Configuracion del Entorno

In [ ]:
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

print("Entorno configurado. Plotly version:", px.__version__ if hasattr(px, '__version__') else 'disponible')

---
## 1. Carga de Datos Agregados desde BigQuery

Cargamos tres conjuntos de datos agregados que serviran como base para todas las visualizaciones:
- **Diario:** viajes, tarifa promedio, propina promedio por dia
- **Por hora:** distribucion horaria de viajes y tarifas
- **Por zona:** metricas por borough estimado

In [ ]:
# Datos agregados diarios
query_daily = """
SELECT
    DATE(pickup_datetime) AS trip_date,
    COUNT(*) AS trips,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(CASE WHEN payment_type = '1' AND fare_amount > 0
              THEN tip_amount / fare_amount END) * 100, 1) AS avg_tip_pct,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(SUM(fare_amount + tip_amount), 2) AS total_revenue
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE fare_amount BETWEEN 2.5 AND 500
  AND trip_distance BETWEEN 0.1 AND 100
  AND pickup_datetime >= '2015-01-01'
  AND pickup_datetime < '2016-01-01'
GROUP BY trip_date
ORDER BY trip_date
"""

df_daily = bq.query(query_daily)
df_daily['trip_date'] = pd.to_datetime(df_daily['trip_date'])
print(f"Datos diarios: {len(df_daily)} dias cargados.")
df_daily.head()

In [ ]:
# Datos agregados por hora y dia de la semana
query_hourly = """
SELECT
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
    EXTRACT(HOUR FROM pickup_datetime) AS hour,
    CASE
        WHEN pickup_location_id IN ('4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263')
        THEN 'Manhattan'
        WHEN pickup_location_id IN ('11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257')
        THEN 'Brooklyn'
        WHEN pickup_location_id = '132' THEN 'JFK Airport'
        WHEN pickup_location_id = '138' THEN 'LaGuardia Airport'
        ELSE 'Queens/Otros'
    END AS zone,
    COUNT(*) AS trips,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE fare_amount BETWEEN 2.5 AND 500
  AND pickup_location_id IS NOT NULL
  AND pickup_datetime >= '2015-01-01'
  AND pickup_datetime < '2016-01-01'
GROUP BY day_of_week, hour, zone
ORDER BY day_of_week, hour
"""

df_hourly = bq.query(query_hourly)
days_map = {1: 'Domingo', 2: 'Lunes', 3: 'Martes', 4: 'Miercoles',
            5: 'Jueves', 6: 'Viernes', 7: 'Sabado'}
df_hourly['day_name'] = df_hourly['day_of_week'].map(days_map)
print(f"Datos por hora: {len(df_hourly)} registros cargados.")

In [ ]:
# Datos agregados por zona
query_zones = """
SELECT
    CASE
        WHEN pickup_location_id IN ('4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263')
        THEN 'Manhattan'
        WHEN pickup_location_id IN ('11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257')
        THEN 'Brooklyn'
        WHEN pickup_location_id = '132' THEN 'JFK Airport'
        WHEN pickup_location_id = '138' THEN 'LaGuardia Airport'
        ELSE 'Queens/Otros'
    END AS zone,
    COUNT(*) AS trips,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(CASE WHEN payment_type = '1' AND fare_amount > 0
              THEN tip_amount / fare_amount END) * 100, 1) AS avg_tip_pct,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND)) / 60.0, 1) AS avg_duration_min
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE fare_amount BETWEEN 2.5 AND 500
  AND trip_distance BETWEEN 0.1 AND 100
  AND pickup_location_id IS NOT NULL
  AND pickup_datetime >= '2015-01-01'
  AND pickup_datetime < '2016-01-01'
GROUP BY zone
ORDER BY trips DESC
"""

df_zones = bq.query(query_zones)
print(f"Datos por zona: {len(df_zones)} zonas cargadas.")
df_zones

---
## 2. Tarjetas de KPI

Panel de indicadores clave de rendimiento con formato visual estilizado.

In [ ]:
# Calcular KPIs globales
total_trips = df_daily['trips'].sum()
avg_daily_trips = df_daily['trips'].mean()
avg_fare = (df_daily['avg_fare'] * df_daily['trips']).sum() / total_trips
avg_tip = df_daily['avg_tip_pct'].mean()
total_revenue = df_daily['total_revenue'].sum()
top_zone = df_zones.iloc[0]['zone']
top_zone_pct = (df_zones.iloc[0]['trips'] / df_zones['trips'].sum() * 100)

kpi_html = f"""
<style>
.dashboard-kpi {{ display: flex; flex-wrap: wrap; gap: 12px; margin: 15px 0; }}
.kpi-box {{
    flex: 1; min-width: 180px; padding: 18px; border-radius: 10px;
    font-family: 'Segoe UI', Arial, sans-serif; text-align: center;
    box-shadow: 0 2px 10px rgba(0,0,0,0.1);
}}
.kpi-box .label {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1px; opacity: 0.8; margin-bottom: 8px; }}
.kpi-box .value {{ font-size: 28px; font-weight: 700; }}
.kpi-box .sub {{ font-size: 11px; opacity: 0.7; margin-top: 4px; }}
</style>

<div class="dashboard-kpi">
  <div class="kpi-box" style="background: linear-gradient(135deg, #667eea, #764ba2); color: white;">
    <div class="label">Total Viajes 2015</div>
    <div class="value">{total_trips:,.0f}</div>
    <div class="sub">{avg_daily_trips:,.0f} viajes/dia</div>
  </div>
  <div class="kpi-box" style="background: linear-gradient(135deg, #11998e, #38ef7d); color: white;">
    <div class="label">Tarifa Promedio</div>
    <div class="value">${avg_fare:.2f}</div>
    <div class="sub">Por viaje</div>
  </div>
  <div class="kpi-box" style="background: linear-gradient(135deg, #ee0979, #ff6a00); color: white;">
    <div class="label">Propina Promedio</div>
    <div class="value">{avg_tip:.1f}%</div>
    <div class="sub">Solo tarjeta de credito</div>
  </div>
  <div class="kpi-box" style="background: linear-gradient(135deg, #4facfe, #00f2fe); color: white;">
    <div class="label">Ingreso Total</div>
    <div class="value">${total_revenue/1e9:.2f}B</div>
    <div class="sub">Tarifa + propina</div>
  </div>
  <div class="kpi-box" style="background: linear-gradient(135deg, #a18cd1, #fbc2eb); color: #333;">
    <div class="label">Zona Dominante</div>
    <div class="value">{top_zone}</div>
    <div class="sub">{top_zone_pct:.0f}% de los viajes</div>
  </div>
</div>
"""

display(HTML(kpi_html))

---
## 3. Mapa de Calor Interactivo: Hora x Dia de la Semana

Este mapa de calor permite visualizar la densidad de viajes por hora y dia de la semana.
Se incluye un filtro por zona mediante un dropdown interactivo.

In [ ]:
# Preparar datos para heatmap con filtro por zona
zones = ['Todas'] + sorted(df_hourly['zone'].unique().tolist())
day_order = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']

fig_heatmap = go.Figure()

# Crear una traza por cada zona (solo la primera visible)
for i, zone in enumerate(zones):
    if zone == 'Todas':
        df_filt = df_hourly.groupby(['day_name', 'hour'])['trips'].sum().reset_index()
    else:
        df_filt = df_hourly[df_hourly['zone'] == zone].groupby(['day_name', 'hour'])['trips'].sum().reset_index()

    pivot = df_filt.pivot(index='day_name', columns='hour', values='trips').reindex(day_order).fillna(0)

    fig_heatmap.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=list(range(24)),
            y=day_order,
            colorscale='YlOrRd',
            name=zone,
            visible=(i == 0),
            colorbar=dict(title='Viajes'),
            hovertemplate='Hora: %{x}<br>Dia: %{y}<br>Viajes: %{z:,.0f}<extra></extra>'
        )
    )

# Crear botones de dropdown
buttons = []
for i, zone in enumerate(zones):
    visibility = [False] * len(zones)
    visibility[i] = True
    buttons.append(dict(
        label=zone,
        method='update',
        args=[{'visible': visibility}]
    ))

fig_heatmap.update_layout(
    title='Mapa de Calor: Viajes por Hora y Dia de la Semana',
    xaxis_title='Hora del dia',
    yaxis_title='Dia de la semana',
    template='plotly_white',
    height=450,
    updatemenus=[dict(
        type='dropdown',
        direction='down',
        x=1.15, y=1.0,
        showactive=True,
        buttons=buttons,
        active=0
    )],
    annotations=[dict(
        text='Zona:', showarrow=False,
        x=1.15, y=1.07, xref='paper', yref='paper',
        font=dict(size=12)
    )]
)

fig_heatmap.show()

---
## 4. Serie Temporal con Pronostico

Viajes diarios con media movil de 7 dias y una franja de pronostico SARIMA.
Se incluye un range slider para explorar periodos especificos.

In [ ]:
# Calcular media movil y simular pronostico SARIMA
df_ts = df_daily[['trip_date', 'trips']].copy()
df_ts['ma_7'] = df_ts['trips'].rolling(window=7, center=True).mean()

# Simular pronostico SARIMA para los ultimos 30 dias
# (En produccion, se usaria el modelo entrenado en Fase 5)
forecast_start = df_ts['trip_date'].max() - pd.Timedelta(days=29)
df_forecast = df_ts[df_ts['trip_date'] >= forecast_start].copy()

# Simular valores de pronostico basados en la media movil con ruido
np.random.seed(42)
df_forecast['forecast'] = df_forecast['ma_7'] * (1 + np.random.normal(0, 0.03, len(df_forecast)))
df_forecast['forecast_upper'] = df_forecast['forecast'] * 1.08
df_forecast['forecast_lower'] = df_forecast['forecast'] * 0.92

fig_ts = go.Figure()

# Viajes reales
fig_ts.add_trace(go.Scatter(
    x=df_ts['trip_date'], y=df_ts['trips'],
    mode='lines', name='Viajes Reales',
    line=dict(color='#667eea', width=1),
    opacity=0.5
))

# Media movil 7 dias
fig_ts.add_trace(go.Scatter(
    x=df_ts['trip_date'], y=df_ts['ma_7'],
    mode='lines', name='Media Movil 7 dias',
    line=dict(color='#e74c3c', width=2.5)
))

# Pronostico SARIMA
fig_ts.add_trace(go.Scatter(
    x=df_forecast['trip_date'], y=df_forecast['forecast'],
    mode='lines', name='Pronostico SARIMA',
    line=dict(color='#2ecc71', width=2.5, dash='dash')
))

# Banda de confianza
fig_ts.add_trace(go.Scatter(
    x=pd.concat([df_forecast['trip_date'], df_forecast['trip_date'][::-1]]),
    y=pd.concat([df_forecast['forecast_upper'], df_forecast['forecast_lower'][::-1]]),
    fill='toself', fillcolor='rgba(46, 204, 113, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Intervalo de Confianza 95%'
))

fig_ts.update_layout(
    title='Viajes Diarios con Media Movil y Pronostico SARIMA',
    xaxis_title='Fecha',
    yaxis_title='Numero de viajes',
    template='plotly_white',
    height=500,
    xaxis=dict(
        rangeslider=dict(visible=True),
        type='date'
    ),
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig_ts.show()

---
## 5. Comparacion por Zona

Graficos de barras agrupadas que comparan viajes, tarifa promedio y propina
entre las principales zonas de operacion. Se incluye un dropdown para seleccionar la metrica.

In [ ]:
# Comparación por zona con dropdown para métricas
df_z = df_zones[df_zones['zone'] != 'Otros'].copy()

metrics = {
    'Viajes (millones)': ('trips', 1e6, ',.1f'),
    'Tarifa Promedio ($)': ('avg_fare', 1, ',.2f'),
    'Propina Promedio (%)': ('avg_tip_pct', 1, ',.1f'),
    'Distancia Promedio (mi)': ('avg_distance', 1, ',.2f'),
    'Duracion Promedio (min)': ('avg_duration_min', 1, ',.1f')
}

fig_zones = go.Figure()
colors = ['#667eea', '#764ba2', '#f093fb', '#4facfe', '#43e97b']

for i, (metric_name, (col, divisor, fmt)) in enumerate(metrics.items()):
    values = df_z[col] / divisor
    fig_zones.add_trace(go.Bar(
        x=df_z['zone'], y=values,
        name=metric_name,
        marker_color=colors,
        text=values.apply(lambda x: f'{x:{fmt}}'),
        textposition='outside',
        visible=(i == 0)
    ))

buttons_zone = []
for i, metric_name in enumerate(metrics.keys()):
    visibility = [False] * len(metrics)
    visibility[i] = True
    buttons_zone.append(dict(
        label=metric_name,
        method='update',
        args=[{'visible': visibility},
              {'yaxis': {'title': metric_name}}]
    ))

fig_zones.update_layout(
    title='Comparacion de Metricas por Zona',
    xaxis_title='Zona',
    template='plotly_white',
    height=500,
    showlegend=False,
    updatemenus=[dict(
        type='dropdown',
        direction='down',
        x=1.05, y=1.0,
        showactive=True,
        buttons=buttons_zone,
        active=0
    )],
    annotations=[dict(
        text='Metrica:', showarrow=False,
        x=1.05, y=1.07, xref='paper', yref='paper',
        font=dict(size=12)
    )]
)

fig_zones.show()

---
## 6. Panel de Anomalias

Visualizacion de la serie temporal con dias anomalos detectados mediante Isolation Forest.
Los dias anomalos se destacan en rojo, permitiendo identificar eventos inusuales
(tormentas, festivos, eventos especiales).

In [ ]:
# Detección de anomalías con Isolation Forest
from sklearn.ensemble import IsolationForest

df_anomaly = df_daily[['trip_date', 'trips', 'avg_fare', 'avg_tip_pct']].dropna().copy()

# Ajustar Isolation Forest
features = ['trips', 'avg_fare', 'avg_tip_pct']
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
df_anomaly['anomaly'] = iso_forest.fit_predict(df_anomaly[features])
df_anomaly['is_anomaly'] = df_anomaly['anomaly'] == -1

df_normal = df_anomaly[~df_anomaly['is_anomaly']]
df_anom = df_anomaly[df_anomaly['is_anomaly']]

print(f"Dias anomalos detectados: {df_anom.shape[0]} de {df_anomaly.shape[0]} ({df_anom.shape[0]/df_anomaly.shape[0]*100:.1f}%)")

fig_anomaly = go.Figure()

# Serie temporal normal
fig_anomaly.add_trace(go.Scatter(
    x=df_anomaly['trip_date'], y=df_anomaly['trips'],
    mode='lines', name='Viajes Diarios',
    line=dict(color='#667eea', width=1.5)
))

# Puntos anomalos
fig_anomaly.add_trace(go.Scatter(
    x=df_anom['trip_date'], y=df_anom['trips'],
    mode='markers', name='Dias Anomalos',
    marker=dict(color='red', size=10, symbol='x', line=dict(width=2)),
    hovertemplate='Fecha: %{x}<br>Viajes: %{y:,.0f}<br><b>ANOMALIA</b><extra></extra>'
))

# Media como referencia
mean_trips = df_anomaly['trips'].mean()
fig_anomaly.add_hline(
    y=mean_trips, line_dash='dash', line_color='gray',
    annotation_text=f'Media: {mean_trips:,.0f}'
)

fig_anomaly.update_layout(
    title='Deteccion de Anomalias con Isolation Forest',
    xaxis_title='Fecha',
    yaxis_title='Viajes diarios',
    template='plotly_white',
    height=450,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig_anomaly.show()

# Listar dias anomalos
print("\nDias anomalos detectados:")
for _, row in df_anom.iterrows():
    print(f"  {row['trip_date'].strftime('%Y-%m-%d')} ({row['trip_date'].strftime('%A')}): {row['trips']:,.0f} viajes")

---
## 7. Mapa de Densidad de Recogidas por Zona

Mapa scatter utilizando Plotly Mapbox con el estilo `open-street-map` (no requiere token).
Muestra la densidad de recogidas por zona TLC, representada con los centroides aproximados
de cada borough/area principal.

In [ ]:
# Consultar datos de pickup agregados por zona (borough)
query_geo = """
SELECT
    CASE
        WHEN pickup_location_id IN ('4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263')
        THEN 'Manhattan'
        WHEN pickup_location_id IN ('11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257')
        THEN 'Brooklyn'
        WHEN pickup_location_id = '132' THEN 'JFK'
        WHEN pickup_location_id = '138' THEN 'LaGuardia'
        ELSE 'Queens/Otro'
    END AS zone,
    COUNT(*) AS trips,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE fare_amount BETWEEN 2.5 AND 500
  AND pickup_location_id IS NOT NULL
  AND pickup_datetime >= '2015-06-01'
  AND pickup_datetime < '2015-06-08'
GROUP BY zone
HAVING COUNT(*) >= 10
ORDER BY trips DESC
"""

df_geo = bq.query(query_geo)

# Asignar coordenadas de centroides por zona
ZONE_CENTROIDS = {
    'Manhattan': (40.7580, -73.9855),
    'Brooklyn': (40.6782, -73.9442),
    'Queens/Otro': (40.7282, -73.7949),
    'JFK': (40.6413, -73.7781),
    'LaGuardia': (40.7769, -73.8740),
}

df_geo['lat'] = df_geo['zone'].map(lambda z: ZONE_CENTROIDS.get(z, (40.7128, -74.0060))[0])
df_geo['lon'] = df_geo['zone'].map(lambda z: ZONE_CENTROIDS.get(z, (40.7128, -74.0060))[1])
df_geo['log_trips'] = np.log10(df_geo['trips'])

print(f"Zonas cargadas: {len(df_geo)}")
df_geo

In [ ]:
# Mapa scatter con open-street-map usando centroides de zona
fig_map = px.scatter_mapbox(
    df_geo,
    lat='lat',
    lon='lon',
    size='trips',
    color='log_trips',
    color_continuous_scale='Hot',
    size_max=40,
    opacity=0.7,
    zoom=10,
    center={'lat': 40.7580, 'lon': -73.9855},
    mapbox_style='open-street-map',
    title='Densidad de Recogidas por Zona (1ra semana junio 2015)',
    labels={'log_trips': 'Log10(Viajes)', 'trips': 'Viajes'},
    hover_name='zone',
    hover_data={'trips': ':,.0f', 'avg_fare': ':$.2f', 'log_trips': False,
                'lat': False, 'lon': False}
)

fig_map.update_layout(
    height=600,
    margin=dict(l=0, r=0, t=40, b=0)
)

fig_map.show()

---
## 8. Dashboard Multi-Panel

Vista consolidada con multiples paneles usando `make_subplots` para obtener
una vision general rapida del estado de las operaciones.

In [ ]:
# Dashboard multi-panel con make_subplots
fig_dash = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Viajes Diarios (2015)',
        'Distribucion por Zona',
        'Distribucion Horaria Promedio',
        'Tarifa vs Propina por Zona'
    ],
    specs=[
        [{'type': 'scatter'}, {'type': 'pie'}],
        [{'type': 'bar'}, {'type': 'bar'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Panel 1: Serie temporal
fig_dash.add_trace(
    go.Scatter(x=df_daily['trip_date'], y=df_daily['trips'],
               mode='lines', line=dict(color='#667eea', width=1),
               name='Viajes'),
    row=1, col=1
)
fig_dash.add_trace(
    go.Scatter(x=df_daily['trip_date'],
               y=df_daily['trips'].rolling(7).mean(),
               mode='lines', line=dict(color='#e74c3c', width=2),
               name='MA(7)'),
    row=1, col=1
)

# Panel 2: Pie chart por zona
fig_dash.add_trace(
    go.Pie(labels=df_zones['zone'], values=df_zones['trips'],
           hole=0.4, marker=dict(colors=['#667eea', '#764ba2', '#f093fb', '#4facfe', '#43e97b']),
           textinfo='label+percent'),
    row=1, col=2
)

# Panel 3: Distribución horaria promedio
df_hourly_avg = df_hourly.groupby('hour')['trips'].mean().reset_index()
fig_dash.add_trace(
    go.Bar(x=df_hourly_avg['hour'], y=df_hourly_avg['trips'],
           marker_color='#667eea', name='Viajes/hora'),
    row=2, col=1
)

# Panel 4: Tarifa vs propina por zona
df_z_plot = df_zones[df_zones['zone'] != 'Otros']
fig_dash.add_trace(
    go.Bar(x=df_z_plot['zone'], y=df_z_plot['avg_fare'],
           name='Tarifa ($)', marker_color='#667eea'),
    row=2, col=2
)
fig_dash.add_trace(
    go.Bar(x=df_z_plot['zone'], y=df_z_plot['avg_tip_pct'],
           name='Propina (%)', marker_color='#2ecc71'),
    row=2, col=2
)

fig_dash.update_layout(
    template='plotly_white',
    height=800,
    title_text='Dashboard Integral de Operaciones - NYC Taxi 2015',
    showlegend=False
)

fig_dash.show()

---
## 9. Exportar Dashboard a HTML

Para compartir el dashboard con stakeholders que no tienen acceso a Jupyter,
podemos exportar cualquiera de las figuras interactivas a un archivo HTML independiente.

### Instrucciones de Exportacion

```python
# Exportar una figura individual
fig_heatmap.write_html('heatmap_interactivo.html', include_plotlyjs=True)

# Exportar el dashboard multi-panel
fig_dash.write_html('dashboard_nyc_taxi.html', include_plotlyjs=True)

# Exportar el mapa
fig_map.write_html('mapa_densidad_pickups.html', include_plotlyjs=True)
```

**Nota:** Al incluir `include_plotlyjs=True`, el archivo HTML es autosuficiente y se puede
abrir en cualquier navegador sin necesidad de conexion a internet. El tamano tipico es de
3-5 MB por archivo.

Para un archivo mas ligero (requiere conexion a internet):
```python
fig_dash.write_html('dashboard_nyc_taxi.html', include_plotlyjs='cdn')
```

In [ ]:
# Exportar dashboard principal a HTML
import os

output_dir = '../../../reports/nyc_taxi/'
os.makedirs(output_dir, exist_ok=True)

export_path = os.path.join(output_dir, 'dashboard_nyc_taxi_2015.html')
fig_dash.write_html(export_path, include_plotlyjs='cdn')
print(f"Dashboard exportado a: {os.path.abspath(export_path)}")

# Exportar mapa
map_path = os.path.join(output_dir, 'mapa_densidad_pickups.html')
fig_map.write_html(map_path, include_plotlyjs='cdn')
print(f"Mapa exportado a: {os.path.abspath(map_path)}")

---
## Conclusion

Este dashboard interactivo proporciona:

1. **KPIs en tiempo real** con tarjetas visuales estilizadas
2. **Mapa de calor** con filtro por zona para analizar patrones temporales
3. **Serie temporal** con pronostico SARIMA y range slider para exploracion
4. **Comparacion por zona** con dropdown para multiples metricas
5. **Panel de anomalias** con deteccion automatica via Isolation Forest
6. **Mapa geografico** de densidad de recogidas sin necesidad de token
7. **Dashboard multi-panel** consolidado
8. **Exportacion a HTML** para compartir con stakeholders

Las visualizaciones son completamente interactivas: se puede hacer zoom, filtrar,
y pasar el cursor sobre los datos para obtener detalles.

---
*Notebook creado como parte de la Fase 6 (Sintesis y Produccion) del proyecto NYC Taxi.*